In [ ]:
!pip install prometheus-eval vllm

: 

In [ ]:
# Install dependensi NVIDIA-nya dulu secara manual
pip install nvidia-cusparselt-cu12 nvidia-nccl-cu12 nvidia-nvjitlink-cu12 nvidia-nvshmem-cu12 --default-timeout=2000

# Jika sudah berhasil, baru install vllm
pip install vllm --default-timeout=2000

# Terakhir prometheus
pip install prometheus-eval --default-timeout=2000

In [ ]:
!pip install PyPDF2

In [ ]:
!pip install bitsandbytes>=0.46.1

In [ ]:
import json
import pandas as pd
from PyPDF2 import PdfReader
from prometheus_eval.vllm import VLLM
from prometheus_eval import PrometheusEval
from prometheus_eval.prompts import ABSOLUTE_PROMPT, SCORE_RUBRIC_TEMPLATE
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os
import glob
import re
from pathlib import Path

# Base paths
splitbook_path = '/kaggle/input/emilia-dataset/splitbook/idn'
dataset_path = '/kaggle/input/emilia-dataset/dataset_fix/idn'
output_base_path = '/kaggle/working/'

# Create output directory if it doesn't exist
os.makedirs(output_base_path, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading translation model...")
trans_model_name = "facebook/nllb-200-distilled-600M"
trans_tokenizer = AutoTokenizer.from_pretrained(trans_model_name)
trans_model = AutoModelForSeq2SeqLM.from_pretrained(trans_model_name).to(device)

def translate_text(text, max_length=512):
    """Translate Indonesian text to English using NLLB-200"""
    if not text or text.strip() == "":
        return ""
    
    # Split long text into chunks
    chunks = [text[i:i+max_length] for i in range(0, len(text), max_length)]
    translated_chunks = []
    
    for chunk in chunks:
        # NLLB-200 specific: Set source language token first
        trans_tokenizer.src_lang = "ind_Latn"  # Set Indonesian as source
        
        # Tokenize input
        inputs = trans_tokenizer(
            chunk, 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=max_length
        ).to(device)
        
        with torch.no_grad():
            translated = trans_model.generate(
                **inputs, 
                max_length=max_length, 
                num_beams=4, 
                early_stopping=True,
                forced_bos_token_id=trans_tokenizer.convert_tokens_to_ids("eng_Latn")  # Force English output
            )
        
        translated_text = trans_tokenizer.decode(translated[0], skip_special_tokens=True)
        translated_chunks.append(translated_text)
    
    return " ".join(translated_chunks)

def extract_page_range(filename):
    """Extract page range from filename"""
    # Look for pattern like "14-24" or similar
    pattern = r'(\d+)-(\d+)'
    match = re.search(pattern, filename)
    if match:
        return f"{match.group(1)}-{match.group(2)}"
    return None

def find_matching_files(folder_name):
    """Find matching PDF and JSON files for a specific folder"""
    # Get PDF files from splitbook folder
    pdf_folder = os.path.join(splitbook_path, folder_name)
    json_folder = os.path.join(dataset_path, folder_name)
    
    if not os.path.exists(pdf_folder) or not os.path.exists(json_folder):
        print(f"Warning: Folder {folder_name} not found in both directories")
        return []
    
    pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))
    json_files = glob.glob(os.path.join(json_folder, "*.json"))
    
    matches = []
    
    for json_file in json_files:
        json_basename = os.path.basename(json_file)
        json_page_range = extract_page_range(json_basename)
        
        if json_page_range:
            # Find matching PDF file with same page range
            for pdf_file in pdf_files:
                pdf_basename = os.path.basename(pdf_file)
                pdf_page_range = extract_page_range(pdf_basename)
                
                if pdf_page_range == json_page_range:
                    matches.append({
                        'pdf_path': pdf_file,
                        'json_path': json_file,
                        'page_range': json_page_range,
                        'folder': folder_name
                    })
                    break
    
    return matches

def process_single_pair(pdf_path, json_path, page_range, folder_name, judge, score_rubric):
    """Process a single PDF-JSON pair"""
    print(f"\nProcessing {folder_name} - Pages {page_range}")
    print(f"PDF: {os.path.basename(pdf_path)}")
    print(f"JSON: {os.path.basename(json_path)}")
    
    # Extract PDF text and translate
    print("Extracting and translating PDF content...")
    reader = PdfReader(pdf_path)
    pages = [page.extract_text() or "" for page in reader.pages]
    context_id = "\n".join(pages)
    
    # Translate context to English (split into smaller chunks for better translation)
    context_chunks = [context_id[i:i+2000] for i in range(0, len(context_id), 2000)]
    translated_context_chunks = []
    
    for i, chunk in enumerate(context_chunks):
        translated_chunk = translate_text(chunk, max_length=400)
        translated_context_chunks.append(translated_chunk)
    
    context_en = "\n".join(translated_context_chunks)
    
    # Load QnA JSON and translate
    print("Loading and translating QnA data...")
    with open(json_path, 'r', encoding='utf-8') as f:
        qna_data = json.load(f)
    
    questions_id = [item['Question'] for item in qna_data]
    outputs_id = [item['Answer'] for item in qna_data]
    
    # Translate questions and answers
    questions_en = []
    outputs_en = []
    
    for i, (question, answer) in enumerate(zip(questions_id, outputs_id)):
        question_en = translate_text(question)
        answer_en = translate_text(answer)
        questions_en.append(question_en)
        outputs_en.append(answer_en)
    
    # Evaluate all QnA
    results = []
    print(f"Starting evaluation of {len(questions_en)} QnA pairs...")
    
    for i, (question_id, answer_id, question_en, answer_en) in enumerate(zip(questions_id, outputs_id, questions_en, outputs_en)):
        
        # Format instruction for answer relevance evaluation
        instruction = f"""You are an expert in NLP evaluation metrics, specially trained to assess answer relevance in responses provided by language models. Evaluate how well the following answer addresses the user's question based on the given document context.

DOCUMENT CONTEXT:
{context_en}

USER'S QUESTION:
{question_en}

Analyze whether the answer directly addresses the user's query, stays on topic, and provides relevant information from the context."""
        
        # Perform single absolute grading
        feedback, score = judge.single_absolute_grade(
            instruction=instruction,
            response=answer_en,
            rubric=score_rubric,
            reference_answer=""  # Not using reference answer
        )
        
        # Convert to 0-5 scale (Prometheus returns direct 0-5 score)
        relevance_score_5 = score
        
        # Convert to 0-1 scale for compatibility with your prompt format
        relevance_score_01 = (score - 1) / 4.0  # Maps 1->0, 5->1
        
        # Save results
        result_row = {
            'folder': folder_name,
            'page_range': page_range,
            'question_id': i + 1,
            'question_id_original': question_id,
            'question_en': question_en,
            'answer_id_original': answer_id,
            'answer_en': answer_en,
            'relevance_score_5': relevance_score_5,
            'relevance_score_01': relevance_score_01,
            'feedback': feedback,
            'is_relevant': relevance_score_5 >= 3  # Consider relevant if score >= 3
        }
        
        results.append(result_row)
    
    return results

# Initialize Prometheus model (do this once)
print("Loading Prometheus evaluation model...")
model = VLLM(
    model="prometheus-eval/prometheus-7b-v2.0",
    dtype=torch.float16,
    trust_remote_code=True,
    quantization="bitsandbytes"
)

# Initialize evaluator with absolute grading template
judge = PrometheusEval(model=model, absolute_grade_template=ABSOLUTE_PROMPT)

# Rubric for answer relevance evaluation (1-5 scale, English)
rubric_data = {
    "criteria": "How well does the AI-generated answer address the user's question and stay relevant to the query based on the given document context?",
    "score1_description": "The answer is completely irrelevant to the user's question, addressing entirely different topics or providing information that has no connection to the query.",
    "score2_description": "The answer is mostly irrelevant with only minimal connection to the user's question, containing significant off-topic information or failing to address the main query.",
    "score3_description": "The answer has moderate relevance, partially addressing the user's question but including substantial irrelevant information or missing key aspects of the query.",
    "score4_description": "The answer is mostly relevant and addresses the main aspects of the user's question, with only minor irrelevant details or slight deviations from the query focus.",
    "score5_description": "The answer is highly relevant and directly addresses all aspects of the user's question, staying completely on-topic and providing focused, appropriate information from the context."
}

score_rubric = SCORE_RUBRIC_TEMPLATE.format(**rubric_data)

# Get all folders from dataset_fix directory
dataset_folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]

print(f"Found {len(dataset_folders)} folders to process: {dataset_folders}")

# Process each folder
for folder_name in dataset_folders:
    
    # Find matching PDF-JSON pairs for this folder
    matches = find_matching_files(folder_name)
    
    if not matches:
        print(f"No matching PDF-JSON pairs found for folder {folder_name}")
        continue
    
    print(f"Found {len(matches)} matching pairs in {folder_name}")
    
    # Create folder-specific output directory
    folder_output_path = os.path.join(output_base_path, folder_name)
    os.makedirs(folder_output_path, exist_ok=True)
    
    all_folder_results = []
    
    # Process each matching pair
    for match in matches:
        try:
            results = process_single_pair(
                match['pdf_path'], 
                match['json_path'], 
                match['page_range'], 
                folder_name, 
                judge, 
                score_rubric
            )
            all_folder_results.extend(results)
            
        except Exception as e:
            print(f"Error processing {match['page_range']}: {str(e)}")
            continue
    
    # Save results for this folder
    if all_folder_results:
        df_results = pd.DataFrame(all_folder_results)
        
        # Display statistics for this folder
        print(f"\n=== ANSWER RELEVANCE EVALUATION STATISTICS FOR {folder_name} ===")
        print(f"Total QnA pairs evaluated: {len(df_results)}")
        
        # Save to CSV
        output_csv = os.path.join(folder_output_path, f"{folder_name}_answer_relevance_evaluation.csv")
        df_results.to_csv(output_csv, index=False, encoding='utf-8')
        print(f"Results saved to: {output_csv}")
    
    else:
        print(f"No results to save for folder {folder_name}")

print("\nCleaning up translation model...")
del trans_model
del trans_tokenizer
torch.cuda.empty_cache()

print("\n" + "="*60)
print("ALL FOLDERS PROCESSED SUCCESSFULLY!")
print("="*60)

In [ ]:
print("Alhamdulillah Answer Relevance Idn kelar")

In [ ]:
!pip install prometheus-eval vllm PyPDF2 bitsandbytes pandas torch transformers pytesseract pdf2image

## IDN-OCR

In [ ]:
import json
import pandas as pd
import pytesseract
from pdf2image import convert_from_path
from prometheus_eval.vllm import VLLM
from prometheus_eval import PrometheusEval
from prometheus_eval.prompts import ABSOLUTE_PROMPT, SCORE_RUBRIC_TEMPLATE
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os
import glob
import re
from pathlib import Path

# ================= KONFIGURASI =================
# Menggunakan OCR Bahasa Indonesia
OCR_LANGUAGE = 'ind' 
# ===============================================

# Base paths
splitbook_path = '/kaggle/input/emilia-dataset/splitbook/idn'
dataset_path = '/kaggle/input/emilia-dataset/dataset_fix/idn'
output_base_path = '/kaggle/working/'

os.makedirs(output_base_path, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading translation model...")
trans_model_name = "facebook/nllb-200-distilled-600M"
trans_tokenizer = AutoTokenizer.from_pretrained(trans_model_name)
trans_model = AutoModelForSeq2SeqLM.from_pretrained(trans_model_name).to(device)

def translate_text(text, max_length=512):
    """Translate Indonesian text to English using NLLB-200"""
    if not text or text.strip() == "":
        return ""
    
    chunks = [text[i:i+max_length] for i in range(0, len(text), max_length)]
    translated_chunks = []
    
    for chunk in chunks:
        trans_tokenizer.src_lang = "ind_Latn"
        inputs = trans_tokenizer(
            chunk, return_tensors="pt", padding=True, truncation=True, max_length=max_length
        ).to(device)
        
        with torch.no_grad():
            translated = trans_model.generate(
                **inputs, max_length=max_length, num_beams=4, early_stopping=True,
                forced_bos_token_id=trans_tokenizer.convert_tokens_to_ids("eng_Latn")
            )
        translated_text = trans_tokenizer.decode(translated[0], skip_special_tokens=True)
        translated_chunks.append(translated_text)
    
    return " ".join(translated_chunks)

def extract_text_with_ocr(pdf_path, lang='ind'):
    """Extract text from PDF using OCR"""
    try:
        # Convert PDF to images
        images = convert_from_path(pdf_path)
        text_content = []
        
        for i, image in enumerate(images):
            # Extract text using Tesseract with Indonesian language
            page_text = pytesseract.image_to_string(image, lang=lang)
            text_content.append(page_text)
            
        return "\n".join(text_content)
    except Exception as e:
        print(f"OCR Error on {pdf_path}: {e}")
        return ""

def extract_page_range(filename):
    pattern = r'(\d+)-(\d+)'
    match = re.search(pattern, filename)
    if match:
        return f"{match.group(1)}-{match.group(2)}"
    return None

def find_matching_files(folder_name):
    pdf_folder = os.path.join(splitbook_path, folder_name)
    json_folder = os.path.join(dataset_path, folder_name)
    
    if not os.path.exists(pdf_folder) or not os.path.exists(json_folder):
        return []
    
    pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))
    json_files = glob.glob(os.path.join(json_folder, "*.json"))
    
    matches = []
    for json_file in json_files:
        json_basename = os.path.basename(json_file)
        json_page_range = extract_page_range(json_basename)
        
        if json_page_range:
            for pdf_file in pdf_files:
                pdf_basename = os.path.basename(pdf_file)
                pdf_page_range = extract_page_range(pdf_basename)
                if pdf_page_range == json_page_range:
                    matches.append({
                        'pdf_path': pdf_file,
                        'json_path': json_file,
                        'page_range': json_page_range
                    })
                    break
    return matches

def process_single_pair(pdf_path, json_path, page_range, folder_name, judge, score_rubric):
    print(f"\nProcessing {folder_name} - Pages {page_range}")
    
    # 1. OCR Process (Indonesian)
    print(f"Extracting PDF content using OCR (Language: {OCR_LANGUAGE})...")
    context_raw = extract_text_with_ocr(pdf_path, lang=OCR_LANGUAGE)
    
    if not context_raw.strip():
        print("Warning: OCR produced empty text.")
        return []

    # 2. Context Translation (Indo -> Eng)
    print("Translating OCR context (Indo) to English...")
    context_chunks = [context_raw[i:i+2000] for i in range(0, len(context_raw), 2000)]
    translated_context_chunks = []
    for chunk in context_chunks:
        translated_context_chunks.append(translate_text(chunk, max_length=400))
    context_en = "\n".join(translated_context_chunks)
    
    # 3. Load & Translate Dataset (Indo -> Eng)
    print("Loading and translating Dataset...")
    with open(json_path, 'r', encoding='utf-8') as f:
        qna_data = json.load(f)
    
    questions_en = []
    outputs_en = []
    questions_id = [item['Question'] for item in qna_data]
    outputs_id = [item['Answer'] for item in qna_data]
    
    for q, a in zip(questions_id, outputs_id):
        questions_en.append(translate_text(q))
        outputs_en.append(translate_text(a))
    
    # 4. Evaluation (Answer Relevance)
    results = []
    print(f"Starting Answer Relevance evaluation of {len(questions_en)} QnA pairs...")
    
    for i, (q_id, a_id, q_en, a_en) in enumerate(zip(questions_id, outputs_id, questions_en, outputs_en)):
        
        # --- MODIFIED: Instruction for Relevance ---
        instruction = f"""You are an expert in NLP evaluation metrics, specially trained to assess answer relevance in responses provided by language models. Evaluate how well the following answer addresses the user's question based on the given document context.

DOCUMENT CONTEXT:
{context_en}

USER'S QUESTION:
{q_en}

Analyze whether the answer directly addresses the user's query, stays on topic, and provides relevant information from the context."""
        
        feedback, score = judge.single_absolute_grade(
            instruction=instruction,
            response=a_en,
            rubric=score_rubric,
            reference_answer=""
        )
        
        relevance_score_5 = score
        relevance_score_01 = (score - 1) / 4.0
        
        results.append({
            'folder': folder_name,
            'page_range': page_range,
            'question_id': i + 1,
            'question_id_original': q_id,
            'question_en': q_en,
            'answer_id_original': a_id,
            'answer_en': a_en,
            'relevance_score_5': relevance_score_5,
            'relevance_score_01': relevance_score_01,
            'feedback': feedback,
            'is_relevant': relevance_score_5 >= 3 # Threshold for relevance
        })
    
    return results

# Initialize Model
print("Loading Prometheus evaluation model...")
model = VLLM(model="prometheus-eval/prometheus-7b-v2.0", dtype=torch.float16, quantization="bitsandbytes")
judge = PrometheusEval(model=model, absolute_grade_template=ABSOLUTE_PROMPT)

# --- MODIFIED: Rubric for Answer Relevance ---
rubric_data = {
    "criteria": "How well does the AI-generated answer address the user's question and stay relevant to the query based on the given document context?",
    "score1_description": "The answer is completely irrelevant to the user's question, addressing entirely different topics or providing information that has no connection to the query.",
    "score2_description": "The answer is mostly irrelevant with only minimal connection to the user's question, containing significant off-topic information or failing to address the main query.",
    "score3_description": "The answer has moderate relevance, partially addressing the user's question but including substantial irrelevant information or missing key aspects of the query.",
    "score4_description": "The answer is mostly relevant and addresses the main aspects of the user's question, with only minor irrelevant details or slight deviations from the query focus.",
    "score5_description": "The answer is highly relevant and directly addresses all aspects of the user's question, staying completely on-topic and providing focused, appropriate information from the context."
}
score_rubric = SCORE_RUBRIC_TEMPLATE.format(**rubric_data)

# Main Loop
dataset_folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]
print(f"Found {len(dataset_folders)} folders.")

for folder_name in dataset_folders:
    matches = find_matching_files(folder_name)
    if not matches: continue
    
    folder_output_path = os.path.join(output_base_path, folder_name)
    os.makedirs(folder_output_path, exist_ok=True)
    all_folder_results = []
    
    for match in matches:
        try:
            results = process_single_pair(match['pdf_path'], match['json_path'], match['page_range'], folder_name, judge, score_rubric)
            all_folder_results.extend(results)
        except Exception as e:
            print(f"Error: {e}")
            
    if all_folder_results:
        df = pd.DataFrame(all_folder_results)
        # --- MODIFIED: Output filename ---
        output_csv = os.path.join(folder_output_path, f"{folder_name}_ocr_idn_answer_relevance_eval.csv")
        df.to_csv(output_csv, index=False)
        print(f"Saved: {output_csv}")

print("Done.")

##  ENG

In [ ]:
import json
import pandas as pd
import pytesseract
from pdf2image import convert_from_path
from prometheus_eval.vllm import VLLM
from prometheus_eval import PrometheusEval
from prometheus_eval.prompts import ABSOLUTE_PROMPT, SCORE_RUBRIC_TEMPLATE
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os
import glob
import re
from pathlib import Path

# ================= KONFIGURASI =================
# Dokumen PDF berbahasa INGGRIS
DOCUMENT_LANGUAGE = 'eng' 
# ===============================================

# Base paths
splitbook_path = '/kaggle/input/emilia-dataset/splitbook/idn' 
dataset_path = '/kaggle/input/emilia-dataset/dataset_fix/idn'
output_base_path = '/kaggle/working/'

os.makedirs(output_base_path, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading translation model...")
trans_model_name = "facebook/nllb-200-distilled-600M"
trans_tokenizer = AutoTokenizer.from_pretrained(trans_model_name)
trans_model = AutoModelForSeq2SeqLM.from_pretrained(trans_model_name).to(device)

def translate_text(text, max_length=512):
    """Translate Indonesian text to English using NLLB-200"""
    if not text or text.strip() == "":
        return ""
    
    chunks = [text[i:i+max_length] for i in range(0, len(text), max_length)]
    translated_chunks = []
    
    for chunk in chunks:
        trans_tokenizer.src_lang = "ind_Latn"
        inputs = trans_tokenizer(
            chunk, return_tensors="pt", padding=True, truncation=True, max_length=max_length
        ).to(device)
        
        with torch.no_grad():
            translated = trans_model.generate(
                **inputs, max_length=max_length, num_beams=4, early_stopping=True,
                forced_bos_token_id=trans_tokenizer.convert_tokens_to_ids("eng_Latn")
            )
        translated_text = trans_tokenizer.decode(translated[0], skip_special_tokens=True)
        translated_chunks.append(translated_text)
    
    return " ".join(translated_chunks)

def extract_text_with_ocr(pdf_path, lang='eng'):
    """Extract text from PDF using OCR"""
    try:
        images = convert_from_path(pdf_path)
        text_content = []
        for i, image in enumerate(images):
            page_text = pytesseract.image_to_string(image, lang=lang)
            text_content.append(page_text)
        return "\n".join(text_content)
    except Exception as e:
        print(f"OCR Error on {pdf_path}: {e}")
        return ""

def extract_page_range(filename):
    pattern = r'(\d+)-(\d+)'
    match = re.search(pattern, filename)
    if match:
        return f"{match.group(1)}-{match.group(2)}"
    return None

def find_matching_files(folder_name):
    pdf_folder = os.path.join(splitbook_path, folder_name)
    json_folder = os.path.join(dataset_path, folder_name)
    
    if not os.path.exists(pdf_folder) or not os.path.exists(json_folder):
        return []
    
    pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))
    json_files = glob.glob(os.path.join(json_folder, "*.json"))
    
    matches = []
    for json_file in json_files:
        json_basename = os.path.basename(json_file)
        json_page_range = extract_page_range(json_basename)
        
        if json_page_range:
            for pdf_file in pdf_files:
                pdf_basename = os.path.basename(pdf_file)
                pdf_page_range = extract_page_range(pdf_basename)
                if pdf_page_range == json_page_range:
                    matches.append({
                        'pdf_path': pdf_file,
                        'json_path': json_file,
                        'page_range': json_page_range
                    })
                    break
    return matches

def process_single_pair(pdf_path, json_path, page_range, folder_name, judge, score_rubric):
    print(f"\nProcessing {folder_name} - Pages {page_range}")
    
    # 1. OCR Process
    print(f"Extracting PDF content using OCR (Language: {DOCUMENT_LANGUAGE})...")
    context_raw = extract_text_with_ocr(pdf_path, lang=DOCUMENT_LANGUAGE)
    
    if not context_raw.strip():
        print("Warning: OCR produced empty text.")
        return []

    # 2. Context Logic (English PDF -> No Translation needed for Context)
    if DOCUMENT_LANGUAGE == 'eng':
        print("Document is English. Skipping context translation.")
        context_en = context_raw
    else:
        # Fallback if you change config later
        print("Document is Indonesian. Translating context to English...")
        context_chunks = [context_raw[i:i+2000] for i in range(0, len(context_raw), 2000)]
        translated_context_chunks = []
        for chunk in context_chunks:
            translated_context_chunks.append(translate_text(chunk, max_length=400))
        context_en = "\n".join(translated_context_chunks)
    
    # 3. Load & Translate Dataset (QnA is Indonesian -> Needs Translation)
    print("Loading and translating Dataset (Questions & Answers)...")
    with open(json_path, 'r', encoding='utf-8') as f:
        qna_data = json.load(f)
    
    questions_en = []
    outputs_en = []
    questions_id = [item['Question'] for item in qna_data]
    outputs_id = [item['Answer'] for item in qna_data]
    
    for q, a in zip(questions_id, outputs_id):
        questions_en.append(translate_text(q))
        outputs_en.append(translate_text(a))
    
    # 4. Evaluation (ANSWER RELEVANCE)
    results = []
    print(f"Starting Answer Relevance evaluation of {len(questions_en)} QnA pairs...")
    
    for i, (q_id, a_id, q_en, a_en) in enumerate(zip(questions_id, outputs_id, questions_en, outputs_en)):
        
        # --- MODIFIED: Instruction for Relevance ---
        instruction = f"""You are an expert in NLP evaluation metrics, specially trained to assess answer relevance in responses provided by language models. Evaluate how well the following answer addresses the user's question based on the given document context.

DOCUMENT CONTEXT:
{context_en}

USER'S QUESTION:
{q_en}

Analyze whether the answer directly addresses the user's query, stays on topic, and provides relevant information from the context."""
        
        feedback, score = judge.single_absolute_grade(
            instruction=instruction,
            response=a_en,
            rubric=score_rubric,
            reference_answer=""
        )
        
        relevance_score_5 = score
        relevance_score_01 = (score - 1) / 4.0
        
        results.append({
            'folder': folder_name,
            'page_range': page_range,
            'question_id': i + 1,
            'question_id_original': q_id,
            'question_en': q_en,
            'answer_id_original': a_id,
            'answer_en': a_en,
            'relevance_score_5': relevance_score_5,
            'relevance_score_01': relevance_score_01,
            'feedback': feedback,
            'is_relevant': relevance_score_5 >= 3 # Threshold for relevance
        })
    
    return results

# Initialize Model
print("Loading Prometheus evaluation model...")
model = VLLM(model="prometheus-eval/prometheus-7b-v2.0", dtype=torch.float16, quantization="bitsandbytes")
judge = PrometheusEval(model=model, absolute_grade_template=ABSOLUTE_PROMPT)

# --- MODIFIED: Rubric for Answer Relevance ---
rubric_data = {
    "criteria": "How well does the AI-generated answer address the user's question and stay relevant to the query based on the given document context?",
    "score1_description": "The answer is completely irrelevant to the user's question, addressing entirely different topics or providing information that has no connection to the query.",
    "score2_description": "The answer is mostly irrelevant with only minimal connection to the user's question, containing significant off-topic information or failing to address the main query.",
    "score3_description": "The answer has moderate relevance, partially addressing the user's question but including substantial irrelevant information or missing key aspects of the query.",
    "score4_description": "The answer is mostly relevant and addresses the main aspects of the user's question, with only minor irrelevant details or slight deviations from the query focus.",
    "score5_description": "The answer is highly relevant and directly addresses all aspects of the user's question, staying completely on-topic and providing focused, appropriate information from the context."
}
score_rubric = SCORE_RUBRIC_TEMPLATE.format(**rubric_data)

# Main Loop
dataset_folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]
print(f"Found {len(dataset_folders)} folders.")

for folder_name in dataset_folders:
    matches = find_matching_files(folder_name)
    if not matches: continue
    
    folder_output_path = os.path.join(output_base_path, folder_name)
    os.makedirs(folder_output_path, exist_ok=True)
    all_folder_results = []
    
    for match in matches:
        try:
            results = process_single_pair(match['pdf_path'], match['json_path'], match['page_range'], folder_name, judge, score_rubric)
            all_folder_results.extend(results)
        except Exception as e:
            print(f"Error: {e}")
            
    if all_folder_results:
        df = pd.DataFrame(all_folder_results)
        # --- MODIFIED: Output filename ---
        output_csv = os.path.join(folder_output_path, f"{folder_name}_ocr_answer_relevance_eval.csv")
        df.to_csv(output_csv, index=False)
        print(f"Saved: {output_csv}")

print("Done.")